## choice

In [ ]:
VERSION = "0.0.0"
PERCENTAGE = 100
STYLE = "pisco_euclid"
SPLIT = "dataset"

## libs

In [ ]:
import resource
import math
import galsim
import hiplot
import holoviews
import numpy
import panel
import tensorflow_datasets as tfds
from astropy.stats import mad_std
from functions_cdfs import (
    compute_centers_and_edges,
    compute_which_bin,
    get_projected,
    get_projected_magnitude,
    load_data_for_inverse_functions,
    old_compute_which_edge_bin,
    old_get_inv_hlr,
    old_get_inv_mags,
    old_get_sersic_index,
)
from functions_filters import get_field_cosmos
from functions_images import GSPARAMS, build_psf, get_constants, get_offset, build_full_image
from functions_lhs import (
    adding_cosmos_catalog_galaxies,
    draw_lhs,
    draw_lhs_gal,
    project_lhs_gal,
    draw_and_project_lhs_img,
)
from functions_redshift import fit_g_vs_z, generate_shear_scaled, redshift_vs_shear_data
from pandas import DataFrame, concat, read_csv
from plots import (
    before_after,
    check_distrib_cosmos,
    check_hlr_sersic_old_new_histo,
    check_image,
    check_lhs_projections,
    check_psf,
    check_single_hlr_sersic_old_new_histo,
    combined_plot,
    comparaison_3d,
    diff_binned_continuous,
    diff_real_param_cosmos,
    heatmaps_inputs,
    histo_galaxies,
    histo_images,
    images_psf,
    interactive_dist,
    interactive_hist,
    interactive_pairplot,
    large_ellps,
    lhs_visu,
    lhs_visu_single,
    noise_shape,
    plot_example,
    sky_coverage,
    style_norma,
    visu_inputs_ouputs,
    show_galsim_image,
    histo_2,
)
from scipy import stats
from utils import (
    DIR_DATASETS,
    DIRS,
    binned_style,
    get_indices_snc,
    get_pairs,
    getting_norms,
    save_psf,
    scale,
    slice_csv,
    check_sampling,
)

holoviews.extension("bokeh", "plotly")
panel.extension()

resource.setrlimit(resource.RLIMIT_NOFILE, (50000, 50000))

DIR_VERSION = DIRS["pisco_euclid"] / VERSION
DIR_SE = DIRS["pisco_euclid_se"] / VERSION
DIR_PLOTS = DIRS["pisco_euclid_plots"] / VERSION
DIR_PLOTS.mkdir(parents=True, exist_ok=True)

## load all params

In [ ]:
dataset, ds_info = tfds.load(
    f"{STYLE}:{VERSION}",
    split=f"{SPLIT}[:{PERCENTAGE}%]",
    data_dir=DIR_DATASETS,
    with_info=True,
    shuffle_files=False,  # no True to match the csv files
)
print(ds_info.metadata)
print(dataset.element_spec.keys())

TOTAL_ELEMENTS = dataset.cardinality().numpy()

hypercube_img = slice_csv(DIR_VERSION / "images_parameters_true.csv", TOTAL_ELEMENTS)
hypercube_gal = slice_csv(DIR_VERSION / "galaxies_parameters_true.csv", TOTAL_ELEMENTS * 10)

keys_dataset = ["image"]
all_params_dataset = dataset.map(lambda x: {key: x[key] for key in keys_dataset})
all_params = list(all_params_dataset.as_numpy_iterator())
images = numpy.stack([item["image"] for item in all_params])

del all_params_dataset, all_params, dataset

# shape noise cancellation

In [ ]:
grouped_values, duplicate_pairs_image = get_indices_snc(hypercube_gal)
duplicate_pairs_image

In [ ]:
CHOICE = 10
apply_arcsinh = True

if len(duplicate_pairs_image) != 0:
    indices = duplicate_pairs_image[CHOICE]
    noise_shape(images[numpy.array(indices)], hypercube_img.iloc[indices], DIR_PLOTS)
    del indices

del apply_arcsinh, CHOICE

# histo hypercubes

In [ ]:
histo_images(hypercube_img, DIR_PLOTS, bins=11)

In [ ]:
histo_galaxies(hypercube_gal, dir_plots=DIR_PLOTS, alpha=0.7, bins=50)

# examples

In [ ]:
print(selected_number := numpy.random.permutation(images.shape[0])[:3])  # numpy.array([6, 8])

selected_images = images[selected_number]
arcsin = True
plot_example(selected_images, DIR_PLOTS, arcsin)
del selected_number, selected_images, arcsin

In [ ]:
visu_inputs_ouputs(dbb={"inputs": hypercube_img, "simu": images}, loggy=True)

# compute stats  (15_000 images max)

In [ ]:
if not (DIR_VERSION / "all_statistics.csv").exists():
    DataFrame(
        [
            {
                "min_images": numpy.amin(images),
                "max_images": numpy.amax(images),
                "mean_images": numpy.mean(images),
                "std_images": mad_std(images),
                "min_avg_g": numpy.amin(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "max_avg_g": numpy.amax(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "min_avg_obs_e": numpy.amin(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "max_avg_obs_e": numpy.amax(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "min_avg_int_e": numpy.amin(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "max_avg_int_e": numpy.amax(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "min_obs_e": numpy.amin(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "max_obs_e": numpy.amax(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "min_int_e": numpy.amin(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
                "max_int_e": numpy.amax(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
            }
        ]
    ).to_csv(DIR_VERSION / "all_statistics.csv", index=False)

# params

In [ ]:
pairs_img = [("avg_g1", "avg_g2"), ("avg_obs_e1", "avg_obs_e2"), ("avg_int_e1", "avg_int_e2")]
pairs_gal = [
    ("int_e1", "int_e2"),
    ("obs_e1", "obs_e2"),
    ("x", "y"),
    ("mag", "hlr"),
    ("mag", "sersic"),
]
all_pairs = get_pairs(pairs_img, hypercube_img, pairs_gal, hypercube_gal)

In [ ]:
alpha = 0.05
size = 2
lhs_visu(DIR_PLOTS, all_pairs, alpha, size)

In [ ]:
NB = 3
alpha = 0.05

lhs_visu_single(DIR_PLOTS, all_pairs[NB], alpha)

# pairplots

In [ ]:
size_dots = 2
columns_drop_img = ["key", "snc", "index"]
interactive_pairplot(
    hypercube_img.drop(columns=columns_drop_img, inplace=False),
    size_dots,
    DIR_PLOTS / "pairplot_images",
    width=1250,
    height=1250,
)

In [ ]:
size_dots = 2
columns_drop_gal = ["int_e1", "int_e2"]
interactive_pairplot(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    size_dots,
    DIR_PLOTS / "pairplot_gal_params",
    width=1350,
    height=1350,
)

# large single ellipticities

In [ ]:
WHICH = "obs"
# WHICH = "int"
THRESH_1 = 0.68
THRESH_2 = 0.02

conditions = (abs(hypercube_gal[f"{WHICH}_e1"]) > THRESH_1) & (
    abs(hypercube_gal[f"{WHICH}_e2"]) < THRESH_2
)

filtered_inputs = hypercube_gal[conditions]
print("cases possible", index_numbers := filtered_inputs.index.tolist())

if len(index_numbers) != 0:
    select_number_e = numpy.random.permutation(numpy.array(index_numbers))[:3]
    large_ellps(
        selected_images=images[select_number_e // 10],
        elps1=hypercube_gal[f"{WHICH}_e1"][select_number_e],
        elps2=hypercube_gal[f"{WHICH}_e2"][select_number_e],
        number=select_number_e // 10,
        directory=DIR_PLOTS,
        name=f"single_{WHICH}",
    )

    del THRESH_1, THRESH_2, WHICH, conditions, filtered_inputs, select_number_e

# norms outputs

In [ ]:
images_norms = getting_norms(data=images)
panel.Tabs(
    ("HISTO", interactive_hist(data=images_norms)), ("DIST", interactive_dist(data=images_norms))
)

# parallel plots

In [ ]:
columns_drop = ["key", "snc", "index"]

_ = hiplot.Experiment.from_dataframe(
    dataframe=concat(
        (images_norms, hypercube_img.drop(columns=columns_drop, inplace=False)), axis=1
    )
).display(force_full_width=True)

# heatmaps

In [ ]:
columns_drop_img = ["key", "snc", "index"]
columns_drop_gal = ["int_e1", "int_e2"]

heatmaps_inputs(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    concat((images_norms, hypercube_img.drop(columns=columns_drop_img, inplace=False)), axis=1),
    DIR_PLOTS,
)

# scaling

In [ ]:
sigma = read_csv(DIR_VERSION / "all_statistics.csv")["std_images"][0]

style_norma(images.flatten(), style="regular", directory=DIR_PLOTS)

In [ ]:
style_norma(
    numpy.arcsinh(images.flatten() / (sigma * 3)), style="arcsinh-sig-3", directory=DIR_PLOTS
)

In [ ]:
NUM = 10

before_after(images[NUM, :, :], DIR_PLOTS)
del NUM

# redshift vs shear


In [ ]:
N_SAMPELS = 1000
hypercube_redshift = draw_lhs(22222, N_SAMPELS, dim=2)
hypercube_redshift = scale(array=hypercube_redshift, low=-0.055, high=0.055)

z_grid = numpy.linspace(0.2, 5, 10000)
z_pdf = z_grid**2 * numpy.exp(-z_grid / 0.8)

z_data, g_data = redshift_vs_shear_data()

deg = 1
shunsheng_fn = fit_g_vs_z(g_data, z_data, deg=deg)

g_magnitude = numpy.sqrt(hypercube_redshift[:, 0] ** 2 + hypercube_redshift[:, 1] ** 2)
z, g1_scaled, g2_scaled = generate_shear_scaled(
    hypercube_redshift[:, 0], hypercube_redshift[:, 1], shunsheng_fn, z_grid, z_pdf, g_magnitude
)

combined_plot(
    DIRS["wip"],
    z,
    z_grid,
    z_pdf,
    z_data,
    g_data,
    shunsheng_fn,
    hypercube_redshift,
    g1_scaled,
    g2_scaled,
)

# check replacing


In [ ]:
NB_OF_GALAXIES = 2_000
PERCENTAGE_COSMOS = 10
version = "9999"
seed_database = 2222

hypercube = draw_lhs_gal(False, seed_database, NB_OF_GALAXIES)
hypercube = project_lhs_gal(hypercube=hypercube, ellips_style="gaussian", pixel_out=False)

In [ ]:
nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_GALAXIES / 100)
rng_local = numpy.random.default_rng(seed=seed_database)
indices = rng_local.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)
indices = numpy.random.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)

check_columns = [0, 1, 4, 5, 6]  # e1 e2 mag hlr sersic
catalog_targets = hypercube[:, check_columns][indices]

filtered_params = read_csv(DIRS["cosmos"] / "filtered_params.csv")
check_columns_cosmos = ["e1", "e2", "mag", "hlr", "sersic"]

catalog_features = filtered_params[check_columns_cosmos].values

results_features = {}
for SCALING in ["min_max", "z_score"]:
    for NORMA in ["euclidean", "mahalanobis"]:
        lhs_gal = adding_cosmos_catalog_galaxies(hypercube, indices, version, SCALING, NORMA)
        results_features[(SCALING, NORMA)] = lhs_gal[numpy.ix_(indices, check_columns)]

In [ ]:
with_all_catalog = False
NB_CHOICES = 10
# numpy.random.seed(22)
choices = numpy.random.choice(nb_galaxies_cosmos, size=NB_CHOICES, replace=False)
choices

comparaison_3d(
    catalog_features, catalog_targets, results_features, choices, DIRS["wip"], with_all_catalog
)

# check old vs new distributions

In [ ]:
NB_OF_GALAXIES = 20_000
hypercube = draw_lhs(seed=22, n_samples=NB_OF_GALAXIES, dim=5)

In [ ]:
inv_cdf_mag_fn = old_get_inv_mags()
mag = get_projected_magnitude(hypercube[:, 0].copy(), inv_cdf_mag_fn)
mag_bin_edges = numpy.arange(20.5, 25, 0.5)
bin_centers = (mag_bin_edges[:-1] + mag_bin_edges[1:]) / 2
which_mag_bin = compute_which_bin(mag, mag_bin_edges, bin_centers)
edge_bin_mags = old_compute_which_edge_bin(mag)
inv_cdfs_hlr_fn = old_get_inv_hlr(edge_bin_mags, which_mag_bin)
hlr = get_projected(hypercube[:, 1].copy(), which_mag_bin, inv_cdfs_hlr_fn)
sersic = old_get_sersic_index(hypercube[:, 2].copy(), edge_bin_mags)

In [ ]:
check_distrib_cosmos(mag, hlr, sersic, DIRS["wip"], "TEST1", bins=100)

In [ ]:
check_lhs_projections(
    DIRS["wip"],
    "TEST1",
    bin_centers,
    mag_bin_edges,
    mag,
    hlr,
    sersic,
    inv_cdfs_hlr_fn,
    inv_cdfs_hlr_fn,
    nb_pdf=50,
    sersic_continue=False,
)

In [ ]:
style = "25.2"
inv_cdf_mag_fn_new, inv_cdfs_hlr_fn_new, inv_cdfs_sersic_fn_new = load_data_for_inverse_functions(
    style
)
mag_bin_edges, bin_centers = compute_centers_and_edges(style, bin_space=0.5)

mag_new = get_projected_magnitude(hypercube[:, 0].copy(), inv_cdf_mag_fn_new)
which_mag_bin = compute_which_bin(mag_new, mag_bin_edges, bin_centers)
hlr_new = get_projected(hypercube[:, 1].copy(), which_mag_bin, inv_cdfs_hlr_fn_new)
sersic_new = get_projected(hypercube[:, 2].copy(), which_mag_bin, inv_cdfs_sersic_fn_new)
ellips = stats.norm.ppf(hypercube[:, 3:5], loc=0, scale=0.26)

In [ ]:
check_distrib_cosmos(mag_new, hlr_new, sersic_new, DIRS["wip"], "TEST2", bins=100)

In [ ]:
check_lhs_projections(
    DIRS["wip"],
    "TEST2",
    bin_centers,
    mag_bin_edges,
    mag_new,
    hlr_new,
    sersic_new,
    inv_cdfs_hlr_fn_new,
    inv_cdfs_sersic_fn_new,
    nb_pdf=30,
    sersic_continue=True,
)

In [ ]:
list_mag, list_hlr, list_ns, list_e1, list_e2 = binned_style(NB_OF_GALAXIES)

In [ ]:
# hc = numpy.stack([mag, hlr, sersic, ellips[:, 0], ellips[:, 1]], axis=1)
hc = numpy.stack([mag_new, hlr_new, sersic_new, ellips[:, 0], ellips[:, 1]], axis=1)
diff_binned_continuous(list_mag, list_hlr, list_ns, list_e1, list_e2, hc, DIRS["wip"])

In [ ]:
colors = ["blue", "green", "red", "purple", "orange", "brown", "pink", "gold", "cyan"]
check_hlr_sersic_old_new_histo(
    DIRS["wip"], mag_bin_edges, mag, hlr, sersic, hlr_new, sersic_new, colors
)

In [ ]:
which = 8
check_single_hlr_sersic_old_new_histo(
    DIRS["wip"], mag_bin_edges, mag, hlr, sersic, hlr_new, sersic_new, which
)

# SE psf

In [ ]:
all_psf = list(DIRS["psf_se"].glob("*.fits"))

for each_psf in all_psf:
    print(each_psf)

CHOICE = 1
print("CHOICE", all_psf[CHOICE].stem)
check_psf(all_psf[CHOICE])

In [ ]:
style = "airy"  #  airy / euclid
psf = build_psf(style)
image_size = 121  # must have odd size for S++
pixel_scale = 0.033

save_psf(DIRS["psf_se"], style, psf=psf, image_size=image_size, pixel_scale=pixel_scale)
del style, psf, image_size, pixel_scale

# SE check assoc


In [ ]:
CHOICE = "000110"
compress = True

check_image(DIR_SE / f"{CHOICE}_simu", DIR_PLOTS, compress)
del CHOICE, compress

# SE sampling


In [ ]:
print(check_sampling(DIR_SE))

# COSMOS test real vs parametric

In [ ]:
CHOICE = 0
SAMPLE = "23.5"
PIXEL_SCALE, PATCH_SIZE, _, EXPTIME, _, _, _, _ = get_constants()

test_cosmos_cat = galsim.COSMOSCatalog(
    sample=SAMPLE,
    dir=DIRS["cosmos"],
    use_real=True,
    exclusion_level="none",
    exptime=EXPTIME,
)

common_kwargs = dict(index=CHOICE, noise_pad_size=0, deep=False, sersic_prec=0.0, gsparams=GSPARAMS)
real_gal = test_cosmos_cat.makeGalaxy(gal_type="real", **common_kwargs)
param_gal = test_cosmos_cat.makeGalaxy(gal_type="parametric", **common_kwargs)


diff_real_param_cosmos(CHOICE, PIXEL_SCALE, PATCH_SIZE, real_gal, param_gal, DIRS["cosmos_plots"])

# COSMOS mollmoll


In [ ]:
fields_23 = get_field_cosmos(sample="23.5")
fields_25 = get_field_cosmos(sample="25.2")
ra_23 = numpy.array(fields_23["RA"])
dec_23 = numpy.array(fields_23["DEC"])
ra_25 = numpy.array(fields_25["RA"])
dec_25 = numpy.array(fields_25["DEC"])

sky_coverage(ra_23, dec_23, ra_25, dec_25, DIRS["cosmos_plots"])

# TEST image


In [ ]:
params = {
    "g1": 0.0,
    "g2": 0.0,
    # "int_e1": numpy.array([0.25, 0.25]),
    # "int_e2": numpy.array([0.7, 0.7]),
    "int_e1": numpy.array([-0.7, 0.7]),
    "int_e2": numpy.array([0.7, -0.7]),
    "off_x": numpy.array([0, 0]),
    "off_y": numpy.array([0, 0]),
    "mag": numpy.array([20.5, 20.5]),
    "hlr": numpy.array([1.5, 1.5]),
    "sersic": numpy.array([0.9, 0.9]),
    "x": numpy.array([150.0, 150.0]),
    "y": numpy.array([150.0, 150.0]),
}

PIXEL_SCALE, PATCH_SIZE, IMG_SIZE, EXPTIME, GAIN, readoutnoise, SKY_BKG, ZP = get_constants()

psf = build_psf("airy")

# observed ellipticities
obs_e1 = params["int_e1"] + params["g1"]
obs_e2 = params["int_e2"] + params["g2"]

# calculate gal flux
flux = EXPTIME / GAIN * 10 ** (-(params["mag"] - ZP) / 2.5)

# galsim images
full_image = build_full_image(IMG_SIZE, PIXEL_SCALE)
noise_image = build_full_image(IMG_SIZE, PIXEL_SCALE)

NB_OBJET = len(params["x"])
for i in range(NB_OBJET):
    # compute Sersic profile with galsim
    gal = galsim.Sersic(
        n=params["sersic"][i],
        half_light_radius=params["hlr"][i],
        flux=flux[i],
        gsparams=GSPARAMS,
        trunc=params["hlr"][i] * 4.5,
    )

    # apply ellips
    gal = gal.shear(e1=params["int_e1"][i], e2=params["int_e2"][i])

    # # rotate
    # ang = 90
    # gal = gal.rotate(ang * galsim.degrees)

    # apply shear
    gal = gal.shear(g1=params["g1"], g2=params["g2"])

    # convolve galaxy with PSF
    final = galsim.Convolve([psf, gal])

    # offset
    image_pos, ix_nominal, iy_nominal, offset = get_offset(
        params["off_x"][i], params["off_y"][i], params["x"][i], params["y"][i]
    )

    # draw galaxy
    stamp = galsim.ImageF(PATCH_SIZE, PATCH_SIZE, scale=PIXEL_SCALE)
    final.drawImage(image=stamp, wcs=full_image.wcs.local(image_pos), offset=offset)
    stamp.setCenter(ix_nominal, iy_nominal)

    # add stamp to the main image
    bounds = stamp.bounds & full_image.bounds
    full_image[bounds] += stamp[bounds]

# gaussian noise image
F_SKY = PIXEL_SCALE ** (2) * EXPTIME * 10 ** (-(SKY_BKG - ZP) / 2.5)  # e-/pixel
variance = (numpy.sqrt(readoutnoise**2 + F_SKY) / GAIN) ** 2  # e- -> ADU
noise = galsim.GaussianNoise(rng=galsim.BaseDeviate(222), sigma=math.sqrt(variance))
noise_image.addNoise(noise)

# add noise
sign = 1
full_image += sign * noise_image

# plot
show_galsim_image(full_image)

# CHECK lhs

In [ ]:
nb_of_samples = 10_000
lhs_img = draw_and_project_lhs_img(seed=2222 * 1_000, n_samples=nb_of_samples)
avg_g = lhs_img[:, :2]
lhs_gal = draw_lhs_gal(False, 2222, nb_of_samples * 10)
# col1 = 0
# col2 = 3
# lhs_gal[:, [col1, col2]] = lhs_gal[:, [col2, col1]]
lhs_gal = project_lhs_gal(lhs_gal, "gaussian", False)
reshaped_per_image = lhs_gal.reshape(-1, 10, 11)
int_ellips_per_image = reshaped_per_image[:, :, :2]
avg_int_ellips_per_image = int_ellips_per_image.mean(axis=1)
obs_ellips_per_image = int_ellips_per_image + avg_g[:, None, :]
avg_obs_ellips_per_image = obs_ellips_per_image.mean(axis=1)

In [ ]:
bins = 50
histo_2(int_ellips_per_image[:, :, 0].flatten(), int_ellips_per_image[:, :, 1].flatten(), bins)
histo_2(obs_ellips_per_image[:, :, 0].flatten(), obs_ellips_per_image[:, :, 1].flatten(), bins)
histo_2(avg_int_ellips_per_image[:, 0], avg_int_ellips_per_image[:, 1], bins)
histo_2(avg_obs_ellips_per_image[:, 0], avg_obs_ellips_per_image[:, 1], bins)

# PSF database

In [ ]:
VERSION_PSF = "0.0.6"
PERCENTAGE_PSF = 100
STYLE_PSF = "psf"
SPLIT_PSF = "dataset"

In [ ]:
dataset, ds_info = tfds.load(
    f"{STYLE_PSF}:{VERSION_PSF}",
    split=f"{SPLIT_PSF}[:{PERCENTAGE_PSF}%]",
    data_dir=DIR_DATASETS,
    with_info=True,
)
print(ds_info.metadata)
print(dataset.element_spec.keys())

keys_dataset = ["image_psf"]
all_params_dataset = dataset.map(lambda x: {key: x[key] for key in keys_dataset})
all_params = list(all_params_dataset.as_numpy_iterator())
image_psf = numpy.stack([item["image_psf"] for item in all_params])

del all_params_dataset, all_params, dataset

In [ ]:
CHOICE = 0
images_psf(DIRS["psf_plots"] / VERSION_PSF, image_psf[CHOICE])